# Calibration Images

This notebook is intended to take a series of calibration images with the Auxiliary Telescope.

Craig Lage - 26Mar21

In [ ]:
import sys
import asyncio
import time
import os
import numpy as np

from lsst.ts import salobj
from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS

In [ ]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [ ]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG
# Make matplotlib less chatty
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [ ]:
#Start classes
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

In [ ]:
# enable components
await atcs.enable({"atdome": "current", "ataos": "current", "athexapod": "current"})
await latiss.enable({"atspectrograph": "current"})

In [ ]:
# If some components fail to enable, some set of commands like the ones below may be needed
# await salobj.set_summary_state(atcs.rem.atdome, salobj.State.ENABLED, settingsToApply='current')
# await latiss.rem.atarchiver.cmd_start.set_start(timeout=30)

In [ ]:
# Now take a bias to make sure everything is working
await latiss.take_bias(1)

In [ ]:
# Take 50 biases seq # 
# Added wait to stop killing the recent images
for i in range(50):
    await asyncio.sleep(2.0)
    await latiss.take_bias(1)

In [ ]:
# Take 10 10 second darks 
await latiss.take_darks(10.0, 10)

In [ ]:
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [ ]:
# This puts the telescope and dome in the correct position with the telescope
# pointing at the dome screen.  It will take several minutes

# MAKE SURE THE DOME SCREEN ILLUMINATOR IS TURNED ON!!!

await atcs.prepare_for_flatfield()

In [ ]:
# Take 10 2 second flats 
await latiss.take_flats(2.0, 10, filter='RG610', grating='empty_1')

In [ ]:
# Take flats for PTC 
# Added wait to stop killing the recent images
for i in range(20):
    exp = 0.2 * float(i+1)
    await latiss.take_flats(exp, 1, filter='RG610', grating='empty_1')
    if exp < 2.0:
        await asyncio.sleep(2.0)
    await latiss.take_flats(exp, 1, filter='RG610', grating='empty_1')


In [ ]:
# If you are done, put things in standby, or shut things down completely
await atcs.standby()
#await atcs.shutdown()

In [ ]:
# Putting latiss back in standby.
await latiss.standby()